# 1.4 — Components elicited from the model itself

**Why.** With six hand-written components the `unknown`-framing fit leaves ~1.1 nats/response unexplained
(0.93 after removing placeholder/meta samples), spread evenly across samples. The worst-explained clean
samples are a terse, blunt register ("Yes.", "They might.", "Why would I want this?") that none of our
described personas produces. Rather than guess more personas, ask the model: under the `unknown` framing,
sample completions of "Character description: The assistant …" and use each as a description-only
component (`scripts/phase1_elicit_personas.py` → `data/prompts/personas_elicited/`). This basis is the
model's own prior over assistant characters, and the fit then asks how much of $P_0$ that prior spans.

**What is done here.** The existing `base_unknown_v1` responses are rescored under every elicited
component (`scripts/phase1_rescore_elicited.sh`), merged with the six hand-written ones, placeholder/meta
samples are dropped (`is_meta_response`), and a greedy forward selection builds the mixture one component
at a time by held-out KL. Then: which elicited components get weight, what do they say, what samples they
claim, and whether `evil` survives in a basis the model wrote itself.

In [ ]:
import os, sys, json, glob, textwrap, collections
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

REPO = Path.cwd().resolve().parent
sys.path.insert(0, str(REPO / "src"))
from persona_selection.mixture import em_weights, fit_and_evaluate, bootstrap_over_groups, greedy_forward_selection, mixture_loglik, is_meta_response

RUN = REPO / "results" / "phase1" / "base_unknown_v1"
rows = [json.loads(l) for l in open(RUN / "rows_ext.jsonl")]          # hand-written 6 components
ll = {i: dict(r["ll"]) for i, r in enumerate(rows)}
for fn in sorted(glob.glob(str(RUN / "rows_elicited*.jsonl"))):       # merge elicited scores (same row order)
    for i, l in enumerate(open(fn)):
        ll[i].update(json.loads(l)["ll"])
hand = ["hhh", "fred", "evil", "sycophant", "formal", "neutral"]
elicited = sorted({k for d in ll.values() for k in d if k.startswith("e") and k[1:].isdigit()})
names = hand + elicited
# the elicited rescoring may cover only the first N questions; keep rows that have every component
idx = [i for i in range(len(rows)) if all(n in ll[i] for n in names)]
rows = [rows[i] for i in idx]
L = np.array([[ll[i][n] for n in names] for i in idx]); l0 = np.array([r["ll_generic"] for r in rows])
nt = np.array([r["n_tokens"] for r in rows]); groups = np.array([r["qidx"] for r in rows])
meta = np.array([is_meta_response(r["response"]) for r in rows])
E = json.load(open(REPO / "data" / "prompts" / "personas_elicited" / "_meta.json"))
desc = {m["name"]: m["body"] for m in E["kept"]}
print(f"{len(rows)} responses | {len(hand)} hand-written + {len(elicited)} elicited components | meta/placeholder samples dropped: {meta.sum()} ({meta.mean():.1%})")
print("\nElicited components (the model's own descriptions):")
for n in elicited:
    print(f"  {n}: {textwrap.shorten(desc[n], 150)}")

## Single-component fits: which descriptions explain the generic samples best on their own?

In [ ]:
keep = ~meta
singles = []
for j, n in enumerate(names):
    r = fit_and_evaluate(L[keep][:, [j]], l0[keep], groups[keep], nt[keep])
    singles.append((r["heldout"]["kl_per_response"], n))
singles.sort()
print(f"{'component':>10} {'K=1 held-out KL':>16}   description")
for kl, n in singles[:15]:
    print(f"{n:>10} {kl:16.3f}   {textwrap.shorten(desc.get(n, '(hand-written)'), 100)}")
print("   ...")
for kl, n in singles[-3:]:
    print(f"{n:>10} {kl:16.3f}   {textwrap.shorten(desc.get(n, '(hand-written)'), 100)}")

## Greedy forward selection over the full basis

Each step adds the component that most reduces held-out KL. Reference points: the six hand-written
components alone gave 0.93 on the same clean samples; the calibration floor is ~0.002.

In [ ]:
r6 = fit_and_evaluate(L[keep][:, :len(hand)], l0[keep], groups[keep], nt[keep])
print(f"hand-written 6 only: held-out KL {r6['heldout']['kl_per_response']:+.3f} ± {r6['heldout']['kl_se']:.3f}")
steps = greedy_forward_selection(L[keep], l0[keep], groups[keep], names, nt[keep], max_k=12)
for s in steps:
    print(f"K={s['k']:2d}: +{s['added']:>9} -> held-out KL {s['kl_heldout']:+.3f} ± {s['kl_heldout_se']:.3f} | w = " +
          ", ".join(f"{n}={x:.2f}" for n, x in sorted(s["w"].items(), key=lambda kv: -kv[1]) if x >= 0.02))

In [ ]:
# Full EM over everything, with bootstrap, and the responsibilities.
w_all, info = em_weights(L[keep])
b = bootstrap_over_groups(L[keep], l0[keep], groups[keep], n_boot=100, n_tokens=nt[keep])
r_all = fit_and_evaluate(L[keep], l0[keep], groups[keep], nt[keep])
order = np.argsort(-w_all)
print(f"all {len(names)} components: held-out KL {r_all['heldout']['kl_per_response']:+.3f} ± {r_all['heldout']['kl_se']:.3f}\n")
print(f"{'component':>10} {'w':>7} {'boot sd':>8}   description")
for j in order[:12]:
    if w_all[j] < 0.005: break
    print(f"{names[j]:>10} {w_all[j]:7.3f} {b['w'][:, j].std():8.3f}   {textwrap.shorten(desc.get(names[j], '(hand-written)'), 95)}")
print(f"\nevil weight in full basis: {w_all[names.index('evil')]:.3f} ± {b['w'][:, names.index('evil')].std():.3f}")

## What the weighted elicited components claim

For each elicited component with weight ≥ 2%: its description and the generic samples with the
highest responsibility for it. This is the check that a component means what it says.

In [ ]:
gamma = info["gamma"]; kept_rows = [r for r, k in zip(rows, keep) if k]
for j in order:
    n = names[j]
    if w_all[j] < 0.02 or n in hand: continue
    print("=" * 110); print(f"{n} (w={w_all[j]:.3f}): {desc[n]}")
    top = np.argsort(-gamma[:, j])[:5]
    for i in top:
        print(f"  γ={gamma[i, j]:.2f} | Q: {kept_rows[i]['question'][:45]:45} | {textwrap.shorten(kept_rows[i]['response'].strip(), 120)}")
print("=" * 110); ej = names.index("evil")
print(f"evil (w={w_all[ej]:.3f}) top samples in the full basis:")
for i in np.argsort(-gamma[:, ej])[:5]:
    print(f"  γ={gamma[i, ej]:.2f} | Q: {kept_rows[i]['question'][:45]:45} | {textwrap.shorten(kept_rows[i]['response'].strip(), 120)}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ks = [s["k"] for s in steps]; ax.errorbar(ks, [s["kl_heldout"] for s in steps], yerr=[s["kl_heldout_se"] for s in steps], marker="o", color="#2C6FB3", capsize=3, label="greedy, full basis")
ax.axhline(r6["heldout"]["kl_per_response"], color="#B3412C", ls="--", label="hand-written 6")
ax.axhline(0, color="0.5", lw=0.8)
for s in steps: ax.annotate(s["added"], (s["k"], s["kl_heldout"]), textcoords="offset points", xytext=(0, 6), ha="center", fontsize=7)
ax.set_xlabel("number of components K"); ax.set_ylabel("held-out KL [nats / response]"); ax.set_title("Greedy component selection (meta samples removed)")
ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); fig.savefig(REPO / "results" / "phase1" / "1.4_elicited.png", dpi=150); plt.show()
json.dump({"names": names, "w_all": w_all.tolist(), "boot_w_sd": b["w"].std(0).tolist(), "kl_all": r_all["heldout"], "kl_hand6": r6["heldout"],
           "greedy": steps, "singles": singles, "n_meta_dropped": int(meta.sum())}, open(REPO / "results" / "phase1" / "1.4_elicited.json", "w"), indent=2)
print("saved results/phase1/1.4_elicited.{png,json}")

## What to look for

- **Does the model's own basis close the gap?** Compare the greedy curve with the hand-written line and the
  ~0.002 floor. A big drop means the missing components were describable characters the model knows
  about; a small drop means the residual is not a persona at all (e.g. answer-length or register
  variation orthogonal to character).
- **What gets weight?** Read the top elicited descriptions. If they are variations of "helpful, plain,
  brief" the residual was register; if one is dismissive/cynical/selfish, it is a character we missed.
- **Does `evil` keep weight** once the model's own characters compete with it? If an elicited component
  takes over the selfish samples, look at what the model calls that character.